# Questão 1 - EDA Orders Dataset [CSV to SQLite]

In [1]:
#Importando as bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3 #Para simular um banco de dados SQL
import csv

In [ ]:
#Criando conexão e banco de dados SQLite
conexao = sqlite3.connect('orders.db')
cur = conexao.cursor()

#Convertendo o arquivo orders .CSV para SQLite
orders = pd.read_csv('Dataset/orders.csv')
orders.to_sql('orders', conexao, if_exists='replace', index=False)

48998

In [24]:
#Primeiro vamos fazer a leitura do arquivo CSV orders na pasta Dataset e ver o conteudo da tabela
#orders = pd.read_csv('Dataset/orders.csv')
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 48998 entries, 0 to 48997
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               48998 non-null  int64  
 1   order_number     48998 non-null  str    
 2   channel          48998 non-null  str    
 3   customer_id      48998 non-null  int64  
 4   salesperson_id   24867 non-null  float64
 5   location_id      48998 non-null  int64  
 6   status           48998 non-null  str    
 7   subtotal         48998 non-null  float64
 8   discount_amount  48998 non-null  float64
 9   total            48998 non-null  float64
 10  placed_at        48998 non-null  str    
 11  created_at       48998 non-null  str    
 12  updated_at       48998 non-null  str    
dtypes: float64(4), int64(3), str(6)
memory usage: 4.9 MB


In [ ]:
#Parte 1 - Visão geral da tabela orders

#Quantidade total de linhas em python
print("Quantidade total de linhas:", orders.shape[0])

#Quantidade total de linhas em SQL
cur.execute('SELECT COUNT(*) FROM orders')
print("Quantidade total de linhas em SQL:", cur.fetchone()[0])

#Quantidade total de colunas em python
print("Quantidade total de colunas:", orders.shape[1])

#Quantidade total de colunas em SQL
cur.execute('SELECT COUNT(*) FROM pragma_table_info("orders")')
print("Quantidade total de colunas em SQL:", cur.fetchone()[0])

#Intervalo de datas analisado (data mínima e máxima) da coluna created_at em python
print("Data mínima:", orders['created_at'].min())
print("Data máxima:", orders['created_at'].max())

#Intervalo de datas analisado (data mínima e máxima) da coluna created_at em SQL
cur.execute('SELECT MIN(created_at) FROM orders')
print("Data mínima em SQL:", cur.fetchone()[0])
cur.execute('SELECT MAX(created_at) FROM orders')
print("Data máxima em SQL:", cur.fetchone()[0])

Quantidade total de linhas: 48998
Quantidade total de linhas em SQL: 48998
Quantidade total de colunas: 13
Quantidade total de colunas em SQL: 13
Data mínima: 2020-01-01 01:19:28
Data máxima: 2026-12-31 23:43:09
Data mínima em SQL: 2020-01-01 01:19:28
Data máxima em SQL: 2026-12-31 23:43:09


In [36]:
#Parte 2 - Análise de valores numéricos

#Valor mínimo da coluna "total" em python
print("Valor mínimo:", orders['total'].min())

#Valor mínimo da coluna "total" em SQL
cur.execute('SELECT MIN(total) FROM orders')
print("Valor mínimo em SQL:", cur.fetchone()[0])

#Valor máximo da coluna "total" em python
print("Valor máximo:", orders['total'].max())

#Valor máximo da coluna "total" em SQL
cur.execute('SELECT MAX(total) FROM orders')
print("Valor máximo em SQL:", cur.fetchone()[0])

#Valor médio da coluna "total" em python
print("Valor médio:", round(orders['total'].mean(), 2))

#Valor médio da coluna "total" em SQL
cur.execute('SELECT AVG(total) FROM orders')
print("Valor médio em SQL:", round(cur.fetchone()[0], 2))

Valor mínimo: 32.62
Valor mínimo em SQL: 32.62
Valor máximo: 127262.02
Valor máximo em SQL: 127262.02
Valor médio: 28704.99
Valor médio em SQL: 28704.99


In [ ]:
#Parte 3 - Interpretação

#Responda de forma resumida:
#Com base na análise exploratória realizada, escreva um breve diagnóstico sobre a confiabilidade da tabela orders para análises futuras.

In [ ]:
#Comente sobre:

#Possíveis outliers em "total":
display(orders['total'].describe())
#Apenas considerando uma rápida avaliação, podemos observar que o valor máximo da coluna "total" 
#é significativamente maior do que a média e o valor mínimo, o que pode indicar a presença de outliers. 
#Seria interessante investigar esses valores para determinar se são erros de entrada de dados ou transações legítimas.

count     48998.000000
mean      28704.992077
std       19425.636818
min          32.620000
25%       13171.235000
50%       25917.840000
75%       40941.882500
max      127262.020000
Name: total, dtype: float64

In [ ]:
#Para isso irei considerar a aplicação do método de IQR (Interquartile Range) para identificar possíveis outliers na coluna "total".
Q1 = orders['total'].quantile(0.25)
Q3 = orders['total'].quantile(0.75)
IQR = Q3 - Q1
#Aplicando coluna de filtro para identificar os outliers
outliers = orders[(orders['total'] < (Q1 - 1.5 * IQR)) | (orders['total'] > (Q3 + 1.5 * IQR))]
print("Quantidade de outliers identificados:", outliers.shape[0])
#Como há possiveis outliers, é recomendável realizar uma análise mais detalhada para determinar se esses valores 
#são válidos ou se devem ser tratados antes de prosseguir com análises futuras.

Quantidade de outliers identificados: 452


In [ ]:
#Qualidade dos dados (valores nulos ou inconsistentes):
#Identificado que possivel coluna salesperson_id é a unica que contem valores nulos e por ser uma coluna de identificação, 
#pode ser que isso apresente um problema de qualidade de dados, pois não é possível identificar o vendedor responsável por uma determinada transação quando é e-commerce.
orders.head()
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 48998 entries, 0 to 48997
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               48998 non-null  int64  
 1   order_number     48998 non-null  str    
 2   channel          48998 non-null  str    
 3   customer_id      48998 non-null  int64  
 4   salesperson_id   24867 non-null  float64
 5   location_id      48998 non-null  int64  
 6   status           48998 non-null  str    
 7   subtotal         48998 non-null  float64
 8   discount_amount  48998 non-null  float64
 9   total            48998 non-null  float64
 10  placed_at        48998 non-null  str    
 11  created_at       48998 non-null  str    
 12  updated_at       48998 non-null  str    
dtypes: float64(4), int64(3), str(6)
memory usage: 4.9 MB


In [ ]:
#E se você considera que o dataset está pronto para análises ou se exigiria tratamento prévio:
#Recomenda-se tratamento prévio dos outilers em total e dos valores nulos em salesperson_id antes de prosseguir com análises futuras, 
#para garantir a confiabilidade dos resultados obtidos.
conexao.close()